# L28 — Common Random Numbers and Variance Reduction

**Module**: M08 | **Chapter**: 11 | **Lecture**: L28

## Learning Objectives
By the end of this notebook you will be able to:
1. Implement Common Random Numbers (CRN) using synchronised random streams.
2. Compare two system configurations using paired-difference t-tests.
3. Measure and interpret the variance reduction achieved by CRN.
4. Identify when CRN helps and when it can fail.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

We compare two pharmacy designs:
- **Config A**: M/M/1 with μ=1.0 (single server)
- **Config B**: M/M/2 with μ=0.6 per server (two slower servers; same arrival rate λ=0.8)
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import simpy
from scipy import stats
from dataclasses import dataclass

## 1. The Key Idea: Synchronised Streams

CRN assigns the same random numbers to corresponding events in both systems.
- The **arrival stream** controls interarrival times — shared between A and B.
- The **service stream** controls service times — shared between A and B.

If Config B is better, it should be better on most random number realisations,
making the difference $D_i = W_q^A(i) - W_q^B(i)$ consistently positive and
with small variance.

In [ ]:
@dataclass
class QueueConfig:
    lam: float
    mu: float
    n_servers: int
    n_customers: int = 2000
    warmup: int = 200
    label: str = ''


def run_queue(cfg: QueueConfig, arr_rng, svc_rng):
    """
    Run the queue with separate arrival and service RNG streams.
    Using separate streams enables synchronisation across configurations.
    """
    env    = simpy.Environment()
    server = simpy.Resource(env, capacity=cfg.n_servers)
    waits  = []

    def customer():
        t0 = env.now
        with server.request() as req:
            yield req
            waits.append(env.now - t0)
            yield env.timeout(svc_rng.exponential(1.0 / cfg.mu))

    def arrivals():
        for _ in range(cfg.n_customers):
            env.process(customer())
            yield env.timeout(arr_rng.exponential(1.0 / cfg.lam))

    env.process(arrivals())
    env.run()
    return np.mean(waits[cfg.warmup:])


CONFIG_A = QueueConfig(lam=0.8, mu=1.0, n_servers=1, label='M/M/1 μ=1.0')
CONFIG_B = QueueConfig(lam=0.8, mu=0.6, n_servers=2, label='M/M/2 μ=0.6')

print(f"Config A: {CONFIG_A.label}  ρ={CONFIG_A.lam/CONFIG_A.mu:.2f}")
print(f"Config B: {CONFIG_B.label}  ρ={CONFIG_B.lam/(CONFIG_B.n_servers*CONFIG_B.mu):.2f}  (ρ per server)")

## 2. CRN: Paired Replications

In [ ]:
N_REPS = 30

wq_a_crn, wq_b_crn = [], []

master = np.random.default_rng(0)

for i in range(N_REPS):
    # Same seed pair for both configurations → CRN
    arr_rng = np.random.default_rng(master.integers(1_000_000))
    svc_rng = np.random.default_rng(master.integers(1_000_000))

    wq_a_crn.append(run_queue(CONFIG_A, arr_rng=arr_rng,
                              svc_rng=np.random.default_rng(svc_rng.bit_generator)))

    # Reset streams to the same starting point for Config B
    wq_b_crn.append(run_queue(CONFIG_B, arr_rng=arr_rng,
                              svc_rng=np.random.default_rng(svc_rng.bit_generator)))

print("Note: the CRN above shares arrival seeds but NOT service seeds between reps.")
print("A stricter implementation would replay the exact same uniform U(0,1) sequence.")
print("We demonstrate the cleaner version below.")

In [ ]:
# Cleaner CRN: fix seeds per replication; both configs use the same seed
def paired_crn(cfg_a, cfg_b, n_reps, base_seed=0):
    """Run n_reps paired replications using CRN."""
    wq_a, wq_b = [], []
    for i in range(n_reps):
        seed = base_seed + i * 1000
        arr_rng_a = np.random.default_rng(seed)
        svc_rng_a = np.random.default_rng(seed + 1)
        arr_rng_b = np.random.default_rng(seed)      # same arrival seed
        svc_rng_b = np.random.default_rng(seed + 1)  # same service seed
        wq_a.append(run_queue(cfg_a, arr_rng_a, svc_rng_a))
        wq_b.append(run_queue(cfg_b, arr_rng_b, svc_rng_b))
    return np.array(wq_a), np.array(wq_b)


def paired_independent(cfg_a, cfg_b, n_reps, base_seed=0):
    """Run n_reps paired replications with INDEPENDENT streams."""
    wq_a, wq_b = [], []
    for i in range(n_reps):
        seed_a = base_seed + i * 1000
        seed_b = base_seed + i * 1000 + 500_000  # different seed block
        wq_a.append(run_queue(cfg_a, np.random.default_rng(seed_a), np.random.default_rng(seed_a+1)))
        wq_b.append(run_queue(cfg_b, np.random.default_rng(seed_b), np.random.default_rng(seed_b+1)))
    return np.array(wq_a), np.array(wq_b)


wq_a_crn, wq_b_crn = paired_crn(CONFIG_A, CONFIG_B, N_REPS, base_seed=0)
wq_a_ind, wq_b_ind = paired_independent(CONFIG_A, CONFIG_B, N_REPS, base_seed=0)

D_crn = wq_a_crn - wq_b_crn
D_ind = wq_a_ind - wq_b_ind

print(f"Mean Wq: A={wq_a_crn.mean():.3f}, B={wq_b_crn.mean():.3f}")
print(f"CRN  differences D=A-B: mean={D_crn.mean():.3f}  std={D_crn.std(ddof=1):.3f}")
print(f"Indp differences D=A-B: mean={D_ind.mean():.3f}  std={D_ind.std(ddof=1):.3f}")
print(f"Variance reduction ratio: {D_ind.var(ddof=1)/D_crn.var(ddof=1):.2f}x")

## 3. Paired-Difference Confidence Interval

In [ ]:
def paired_ci(D, alpha=0.05, label=''):
    n  = len(D)
    m  = D.mean()
    s  = D.std(ddof=1)
    tc = stats.t.ppf(1 - alpha/2, df=n-1)
    h  = tc * s / np.sqrt(n)
    print(f"{label}: D̄ = {m:.3f}  s_D = {s:.3f}  95% CI = [{m-h:.3f}, {m+h:.3f}]")
    conclusion = "Config B significantly better" if m - h > 0 else "No significant difference"
    print(f"  → {conclusion}")
    return m, h

m_crn, h_crn = paired_ci(D_crn, label='CRN   ')
m_ind, h_ind = paired_ci(D_ind, label='Indep ')

print(f"\nCRN half-width {h_crn:.3f} vs Independent {h_ind:.3f}")
print(f"CRN CI is {h_ind/h_crn:.2f}× narrower")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (D, label, col) in zip(axes, [
    (D_crn, f'CRN  (s_D={D_crn.std(ddof=1):.3f})', 'steelblue'),
    (D_ind, f'Indep (s_D={D_ind.std(ddof=1):.3f})', 'tomato'),
]):
    ax.scatter(np.arange(1, N_REPS+1), D, color=col, s=30, zorder=3)
    ax.axhline(D.mean(), color=col, lw=2, label=f'D̄={D.mean():.3f}')
    ax.axhline(0, color='black', lw=1, linestyle='--')
    ax.set_xlabel('Replication')
    ax.set_ylabel('D = Wq(A) − Wq(B)')
    ax.set_title(f'{label}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Paired differences: CRN vs Independent streams', fontsize=11)
plt.tight_layout()
plt.show()

## 4. When CRN Fails: Positive Correlation Not Guaranteed

CRN requires that the same random number produces the same *direction* of change
in both systems. If the mapping from U to outputs differs structurally, CRN can
induce negative correlation — *increasing* variance.

In [ ]:
# Compute the cross-replication correlation Corr(Wq_A, Wq_B)
r_crn = np.corrcoef(wq_a_crn, wq_b_crn)[0, 1]
r_ind = np.corrcoef(wq_a_ind, wq_b_ind)[0, 1]

print(f"Correlation Wq_A vs Wq_B (CRN):       r = {r_crn:.3f}")
print(f"Correlation Wq_A vs Wq_B (Independent): r = {r_ind:.3f}")
print()
print("Var(D) = Var(A) + Var(B) - 2*Cov(A,B)")
print("CRN reduces Var(D) only if Cov(A,B) > 0 (positive correlation).")
print("If the two systems respond oppositely to the same RN, CRN can INCREASE variance.")

# Theoretical prediction
var_a = wq_a_crn.var(ddof=1)
var_b = wq_b_crn.var(ddof=1)
cov_ab = np.cov(wq_a_crn, wq_b_crn, ddof=1)[0, 1]
var_D_predicted = var_a + var_b - 2 * cov_ab
print(f"\nVar(D) predicted: {var_D_predicted:.4f}")
print(f"Var(D) actual:    {D_crn.var(ddof=1):.4f}")

## 5. Three-Way Comparison with CRN

Extend to three configurations: M/M/1 (μ=1.0), M/M/2 (μ=0.6), M/M/2 (μ=0.7).

In [ ]:
CONFIG_C = QueueConfig(lam=0.8, mu=0.7, n_servers=2, label='M/M/2 μ=0.7')

wq_configs = {'A (M/M/1 μ=1.0)': [], 'B (M/M/2 μ=0.6)': [], 'C (M/M/2 μ=0.7)': []}
cfgs = [CONFIG_A, CONFIG_B, CONFIG_C]

for i in range(N_REPS):
    seed = 10_000 + i * 1000
    for cfg, key in zip(cfgs, wq_configs):
        arr_rng = np.random.default_rng(seed)
        svc_rng = np.random.default_rng(seed + 1)
        wq_configs[key].append(run_queue(cfg, arr_rng, svc_rng))

fig, ax = plt.subplots(figsize=(8, 4))

results_list = []
for key, wqs in wq_configs.items():
    arr = np.array(wqs)
    m = arr.mean()
    h = stats.t.ppf(0.975, df=N_REPS-1) * arr.std(ddof=1) / np.sqrt(N_REPS)
    results_list.append({'label': key, 'mean': m, 'h': h})

labels_plot = [r['label'] for r in results_list]
means_plot  = [r['mean'] for r in results_list]
hs_plot     = [r['h'] for r in results_list]

x = np.arange(len(labels_plot))
ax.bar(x, means_plot, yerr=hs_plot, color=['steelblue','tomato','seagreen'],
       alpha=0.7, capsize=8)
ax.set_xticks(x); ax.set_xticklabels(labels_plot, fontsize=9)
ax.set_ylabel('Mean Wq (min)')
ax.set_title(f'Three-way comparison with CRN ({N_REPS} reps)')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

for r in results_list:
    print(f"{r['label']:25s}: Wq = {r['mean']:.3f} ± {r['h']:.3f} min")

---
## Try It Yourself

1. **Antithetic variates**: Another variance reduction technique uses pairs of uniform samples U and 1-U. Modify `run_queue` to accept a flag `antithetic=True` that inverts the service time variates (i.e., uses 1/µ·ln(1/(1-U)) instead of the standard exponential). Estimate Wq with and without antithetic variates (30 pairs each). Compare the standard deviation of the mean estimate.

2. **CRN synchronisation failure**: Compare two M/M/1 systems with different arrival rates (λ=0.6 vs λ=0.8) using CRN on the service stream only. Plot the correlation between the two Wq estimates. Does CRN help here? Why or why not?

3. **Multiple comparison correction**: With three configurations, there are 3 pairwise comparisons. If each test uses α=0.05, the family-wise Type I error is > 0.05. Apply the Bonferroni correction (α/3 per test) and recompute the significance of each pairwise difference. Does the conclusion change?